# مكتبة مجسّمات Ayso — توليد مجاني ١٠٠٪ على Google Colab### نص ← صورة ← مجسّم ثلاثي الأبعاد · بدون كارت بنكي · بدون GPU**الخطوات:**1. `Runtime → Change runtime type → T4 GPU` (مهم!)2. شغّل الخلايا بالترتيب: ١ → ٢ → (الخلايا تنهار، هذا طبيعي) → ٣ → ٤ → ٥ → ٦3. آخر خلية تعطيك ملف `ayso-library.zip` — نزّله وأرسله لي**الوقت المتوقع:** ~٩٠ ثانية لكل مجسّم · ٩٦ مجسّم ≈ ٢.٥ ساعة**الذاكرة:** ٤.٦ جيجا فقط (من أصل ١٥ متاحة) — مريح جداً

In [ ]:
# ١) التثبيت — يحافظ على torch الموجود بكولاب!pip -q install hy3dgen --no-deps!pip -q install omegaconf einops trimesh pygltflib pyyaml safetensors opencv-python-headless rembg onnxruntime scikit-image pymeshlab fast-simplification!pip -q install 'transformers>=4.49,<5' 'diffusers>=0.33,<0.36'print("تم التثبيت ✓ — انتقل للخلية ٢")

In [ ]:
# ٢) إعادة تشغيل النواة (ضروري) — كولاب سيقول "crashed"، هذا طبيعي. اكمل للخلية ٣import os; os.kill(os.getpid(), 9)

In [ ]:
# ٣) تحميل نموذج المجسّمات (٨ جيجا مرة واحدة، يُخزَّن بالدرايف إن أردت)import osUSE_DRIVE = TrueMODEL_DIR = "/content/hunyuan-mini"try:    if USE_DRIVE:        from google.colab import drive        drive.mount('/content/drive')        MODEL_DIR = "/content/drive/MyDrive/colab-models/hunyuan3d-2mini"except Exception as e:    print("الدرايف غير متاح، نستخدم القرص المؤقت:", e)if not os.path.isdir(os.path.join(MODEL_DIR, "hunyuan3d-dit-v2-mini-turbo")):    os.makedirs(MODEL_DIR, exist_ok=True)    from huggingface_hub import snapshot_download    snapshot_download("tencent/Hunyuan3D-2mini", local_dir=MODEL_DIR,                      allow_patterns=["hunyuan3d-dit-v2-mini-turbo/*"])print("النموذج جاهز في", MODEL_DIR)

In [ ]:
# ٤) توليد الصور من النصوص — نموذج SD-Turbo مجاني وسريع على T4import torch, osfrom PIL import Imagefrom diffusers import AutoPipelineForText2ImageOUT = "/content/ayso"os.makedirs(OUT + "/imgs", exist_ok=True)os.makedirs(OUT + "/glb", exist_ok=True)sd = AutoPipelineForText2Image.from_pretrained(    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")sd.set_progress_bar_config(disable=True)PROMPTS = [{"s": "chair","e": "a simple wooden chair with four legs and a backrest","a": "كرسي"},{"s": "table","e": "a rectangular wooden dining table with four legs","a": "طاولة"},{"s": "house","e": "a small house with a red pitched roof and a door","a": "بيت"},{"s": "tree","e": "a low poly tree with a thick trunk and round canopy","a": "شجرة"},{"s": "car","e": "a compact sedan car","a": "سيارة"},{"s": "cup","e": "a ceramic coffee cup","a": "كأس"},{"s": "rocket","e": "a space rocket with fins","a": "صاروخ"},{"s": "sword","e": "a long sword with a handle and crossguard","a": "سيف"},{"s": "lamp","e": "a table lamp with a conical shade","a": "مصباح"},{"s": "tower","e": "a tall stone tower","a": "برج"},{"s": "cube","e": "a simple cube box","a": "مكعب"},{"s": "ball","e": "a sphere ball","a": "كرة"},{"s": "door","e": "a wooden door with a frame","a": "باب"},{"s": "bridge","e": "a stone arch bridge","a": "جسر"},{"s": "pencil","e": "a pencil","a": "قلم"},{"s": "clock","e": "a wall clock with hands and numbers","a": "ساعة"},{"s": "bed","e": "a bed with a mattress and headboard","a": "سرير"},{"s": "sofa","e": "a three seat sofa","a": "كنبة"},{"s": "bicycle","e": "a bicycle with two wheels and a frame","a": "دراجة"},{"s": "airplane","e": "a small propeller airplane","a": "طائرة"},{"s": "boat","e": "a small wooden rowing boat","a": "قارب"},{"s": "robot","e": "a friendly humanoid robot","a": "روبوت"},{"s": "castle","e": "a medieval castle with towers","a": "قصر"},{"s": "dragon","e": "a stylized dragon with wings","a": "تنين"},{"s": "slide","e": "a children playground slide","a": "زحليقة"},{"s": "truck","e": "a cargo truck with a cab and trailer","a": "شاحنة"},{"s": "train","e": "a steam locomotive train","a": "قطار"},{"s": "ship","e": "a sailing ship with masts","a": "سفينة"},{"s": "mosque","e": "a mosque with a dome and two minarets","a": "مسجد"},{"s": "tv","e": "a flat screen television on a stand","a": "تلفاز"},{"s": "fridge","e": "a refrigerator","a": "ثلاجة"},{"s": "wardrobe","e": "a wooden wardrobe with two doors","a": "دولاب"},{"s": "helicopter","e": "a helicopter with a main rotor","a": "مروحية"},{"s": "tank","e": "a military tank with tracks and a cannon","a": "دبابة"},{"s": "mirror","e": "an oval standing mirror","a": "مرآة"},{"s": "basket","e": "a woven wicker basket","a": "سلة"},{"s": "ladder","e": "a wooden ladder","a": "سلم"},{"s": "computer","e": "a desktop computer monitor and keyboard","a": "حاسوب"},{"s": "phone","e": "a smartphone","a": "هاتف"},{"s": "book","e": "a closed hardcover book","a": "كتاب"},{"s": "rock","e": "a large gray boulder","a": "صخرة"},{"s": "mountain","e": "a rocky mountain peak","a": "جبل"},{"s": "cloud","e": "a fluffy cloud","a": "سحابة"},{"s": "sun","e": "a stylized sun with rays","a": "شمس"},{"s": "moon","e": "a crescent moon","a": "قمر"},{"s": "star","e": "a five pointed star","a": "نجمة"},{"s": "ring","e": "a golden ring","a": "حلقة"},{"s": "pyramid","e": "an egyptian pyramid","a": "هرم"},{"s": "cylinder","e": "a metal cylinder","a": "اسطوانة"},{"s": "plank","e": "a wooden plank","a": "لوح"},{"s": "key","e": "an old metal key","a": "مفتاح"},{"s": "arrow","e": "an arrow with fletching","a": "سهم"},{"s": "flower","e": "a flower with petals and stem","a": "زهرة"},{"s": "fish","e": "a small fish","a": "سمكة"},{"s": "bird","e": "a small bird","a": "طائر"},{"s": "cat","e": "a sitting cat","a": "قطة"},{"s": "dog","e": "a standing dog","a": "كلب"},{"s": "horse","e": "a standing horse","a": "حصان"},{"s": "camera","e": "a photo camera","a": "كاميرا"},{"s": "guitar","e": "an acoustic guitar","a": "قيثارة"},{"s": "piano","e": "an upright piano","a": "بيانو"},{"s": "barrel","e": "a wooden barrel","a": "برميل"},{"s": "fence","e": "a wooden fence section","a": "سياج"},{"s": "well","e": "a stone water well","a": "بئر"},{"s": "tent","e": "a camping tent","a": "خيمة"},{"s": "bus","e": "a city bus","a": "باص"},{"s": "motorcycle","e": "a motorcycle","a": "دراجة نارية"},{"s": "fountain","e": "a stone fountain","a": "نافورة"},{"s": "statue","e": "a marble statue of a person","a": "تمثال"},{"s": "vase","e": "a ceramic vase","a": "مزهرية"},{"s": "candle","e": "a burning candle","a": "شمعة"},{"s": "bottle","e": "a glass bottle","a": "قنينة"},{"s": "globe","e": "a world globe on a stand","a": "كرة أرضية"},{"s": "drum","e": "a drum","a": "طبل"},{"s": "umbrella","e": "an open umbrella","a": "مظلة"},{"s": "bench","e": "a park bench","a": "مقعد"},{"s": "stairs","e": "a short staircase","a": "درج"},{"s": "fountain_pen","e": "a fountain pen","a": "قلم حبر"},{"s": "teddy","e": "a teddy bear","a": "دبدوب"},{"s": "crown","e": "a golden crown","a": "تاج"},{"s": "shield","e": "a round shield","a": "درع"},{"s": "axe","e": "an axe with a wooden handle","a": "فأس"},{"s": "hammer","e": "a hammer","a": "مطرقة"},{"s": "shovel","e": "a shovel","a": "مجرفة"},{"s": "bucket","e": "a metal bucket","a": "دلو"},{"s": "suitcase","e": "a travel suitcase","a": "حقيبة"},{"s": "helmet","e": "a motorcycle helmet","a": "خوذة"},{"s": "goggles","e": "swimming goggles","a": "نظارة"},{"s": "shoe","e": "a leather shoe","a": "حذاء"},{"s": "hat","e": "a wide brim hat","a": "قبعة"},{"s": "glove","e": "a leather glove","a": "قفاز"},{"s": "watch","e": "a wrist watch","a": "ساعة يد"},{"s": "keyboard","e": "a computer keyboard","a": "لوحة مفاتيح"},{"s": "mouse","e": "a computer mouse","a": "فأرة"},{"s": "speaker","e": "a bluetooth speaker","a": "مكبر صوت"},{"s": "printer","e": "an office printer","a": "طابعة"}]SUFFIX = ", single object, centered, full view, product photo, plain white background, 3d render"for it in PROMPTS:    p = f"{OUT}/imgs/{it['s']}.png"    if os.path.exists(p): continue    img = sd(prompt=it['e'] + SUFFIX, num_inference_steps=2, guidance_scale=0.0, height=512, width=512).images[0]    img.save(p)print("الصور جاهزة:", len(os.listdir(OUT + '/imgs')))

In [ ]:
# ٥) تحويل الصور لمجسّمات — الحلقة الرئيسية# يمكن تقسيمها: عدّل START و END (مثلاً 0 و 24) إذا انتهت الجلسةimport time, torch, os, glob, trimeshfrom PIL import Imagefrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipelinepipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(    MODEL_DIR, subfolder='hunyuan3d-dit-v2-mini-turbo', use_safetensors=True, device='cuda')START, END = 0, 200FACE_TARGET = 4000          # خفيف للموقع (4000 مثلث ≈ 80 كيلو)imgs = sorted(glob.glob(OUT + "/imgs/*.png"))[START:END]t0 = time.time()for i, path in enumerate(imgs):    slug = os.path.basename(path).replace(".png", "")    dst = f"{OUT}/glb/{slug}.glb"    if os.path.exists(dst): continue    try:        img = Image.open(path).convert("RGBA")        if img.getextrema()[3][0] == 255:            from hy3dgen.rembg import BackgroundRemover            img = BackgroundRemover()(img)        mesh = pipe(image=img, num_inference_steps=5, octree_resolution=320,                    num_chunks=20000, generator=torch.manual_seed(12345), output_type='trimesh')[0]        mesh = mesh.simplify_quadric_decimation(face_count=FACE_TARGET)        mesh.export(dst)        print(f"[{i+1}/{len(imgs)}] {slug} ✓ {len(mesh.faces)} مثلث · {(time.time()-t0)/ (i+1):.0f}ث/مجسّم", flush=True)    except Exception as e:        print(f"[{i+1}] {slug} ✗ {str(e)[:120]}", flush=True)    finally:        torch.cuda.empty_cache()print("خلصت. المجسّمات:", len(glob.glob(OUT + '/glb/*.glb')))

In [ ]:
# ٦) ضغط النتائج وتنزيلهاimport shutil, osshutil.make_archive("/content/ayso-library", 'zip', OUT + "/glb")print("الحجم:", round(os.path.getsize("/content/ayso-library.zip")/1048576, 2), "ميجا")from google.colab import filesfiles.download("/content/ayso-library.zip")